# How does ITCA behave when the inputs change?

**A small diagnostic, not the paper's benchmark.** We vary label permutation, cell count and evaluation seed separately. The molecular representation stays fixed: this checks conditional ITCA behavior, not the robustness of the full preprocessing/clustering workflow.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gauge_sc as gauge

adata = gauge.datasets.audit_example()
group = adata.obs["group"].astype(str)
adata.obs["labels"] = group.where(~group.str.startswith("Q"), "Q")
views = {"molecular": {"obsm": "X_pca"}}
original_labels = adata.obs["labels"].to_numpy().copy()
order = np.random.default_rng(0).permutation(adata.n_obs)

## 1. Permute labels; keep expression and group sizes fixed

Shuffle labels within a seeded subset. Requested fractions describe the subset, not the fraction whose label actually changes. We report both. Each altered assignment is fixed before evaluation.

In [ ]:
label_rows, label_results, label_inputs = [], {}, {}
for fraction in [0.0, 0.25, 0.5]:
    changed = adata.copy()
    labels = original_labels.copy()
    indices = order[:round(fraction * adata.n_obs)]
    labels[indices] = np.random.default_rng(1).permutation(labels[indices])
    changed.obs["labels"] = labels
    result = gauge.evaluate(changed, groupby="labels", views=views, random_state=0)
    label_inputs[fraction], label_results[fraction] = changed, result
    label_rows.append({
        "Shuffled subset": fraction,
        "Actually changed": np.mean(labels != original_labels),
        "ITCA": result.table.itca.iloc[0],
        "Entropy": result.table.entropy.iloc[0],
    })
label_summary = pd.DataFrame(label_rows)
with sns.plotting_context("paper"), plt.rc_context({
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black", "axes.grid": False,
}):
    fig, ax = plt.subplots(figsize=(4.8, 3.2), layout="constrained")
    ax.plot(label_summary["Shuffled subset"], label_summary.ITCA,
            marker="o", color="#1f77b4", label="ITCA")
    ax.axhline(label_summary.Entropy.iloc[0], color="black", linestyle="--", label="Entropy ceiling")
    ax.set(xlabel="Fraction included in label shuffle", ylabel="ITCA (nats)",
           xticks=[0, 0.25, 0.5], ylim=(0, label_summary.Entropy.max() * 1.12))
    ax.legend(frameon=False)
    sns.despine(ax=ax)
    plt.show()
label_summary.round(3)

## 2. Reduce cells; retain all observed groups

Use nested uniform subsets, without balancing labels or choosing a favorable sample. Proportions are recomputed. Too few cells in a group makes fivefold evaluation unavailable; that is not a zero ITCA score. Audit eligibility additionally requires five cells per test fold.

Fewer cells need not lower ITCA: sampling also changes population proportions.

In [ ]:
sample_rows, sample_results, sample_inputs = [], {}, {}
for n_cells in [560, 140, 20]:
    sample = adata[order[:n_cells]].copy()
    sample_inputs[n_cells] = sample
    row = {"Cells": n_cells, "Groups present": sample.obs.labels.nunique()}
    try:
        result = (label_results[0.0] if n_cells == adata.n_obs else
                  gauge.evaluate(sample, groupby="labels", views=views, random_state=0))
    except ValueError as error:
        if "INSUFFICIENT_GROUP_SIZE" not in str(error):
            raise
        row.update({"ITCA": np.nan, "Audit-eligible groups": None,
                    "Status": "Insufficient cells per group for CV"})
    else:
        sample_results[n_cells] = result
        row.update({"ITCA": result.table.itca.iloc[0],
                    "Audit-eligible groups": int((result.groups.min_test_cells >= 5).sum()),
                    "Status": "Evaluated"})
    sample_rows.append(row)
sample_summary = pd.DataFrame(sample_rows)
sample_summary["Audit-eligible groups"] = pd.array(
    sample_summary["Audit-eligible groups"], dtype="Int64"
)
sample_summary.round(3).astype(object).where(sample_summary.notna(), "—")

## 3. Change the evaluation seed on the same perturbed labels

Seeds 0–2 vary the CV split and default random forest together. This describes seed sensitivity, not a confidence interval or classifier comparison. It cannot detect bias shared by all runs.

In [ ]:
seed_results = {0: label_results[0.25]}
for seed in [1, 2]:
    seed_results[seed] = gauge.evaluate(
        label_inputs[0.25], groupby="labels", views=views, random_state=seed
    )
seed_summary = pd.DataFrame([
    {"Seed": seed, "ITCA": result.table.itca.iloc[0]}
    for seed, result in seed_results.items()
])
seed_summary.round(3)

Less support after label permutation is expected here; individual finite-sample runs need not be monotone. Cell subsets change the evaluated population, so their scores are diagnostics, not candidate rows to combine in `select()`.

For a real workflow stress test, perturb upstream inputs and rerun its preprocessing and candidate-generation steps explicitly. This notebook does not simulate sequencing-depth loss, domain shift or certify robustness. [Methods and optional classifier checks](../docs/recipes.md#controlled-stress-checks).

## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to explore which input changes would meaningfully stress this grouping evaluation. Choose a small set of perturbations and compute settings that address the question, explain what each curve measures, and keep this diagnostic distinct from a full benchmark.

These tutorial settings are an example; adapt them to your question and data.